In [1]:
import tensorflow as tf
import tensorflow_hub as hub

model = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
movenet = model.signatures["serving_default"]

c:\Users\BLESSED-PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.3.0)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


In [5]:
import cv2

In [4]:
#Save model
tf.saved_model.save(model, "movenet_model", signatures=model.signatures["serving_default"])

INFO:tensorflow:Assets written to: movenet_model\assets


INFO:tensorflow:Assets written to: movenet_model\assets


PRE-PROCESSING


In [6]:
def preprocess_image(image):
    image = tf.image.resize(image, (192, 192))
    image = tf.cast(image, dtype=tf.int32)
    image = tf.expand_dims(image, axis=0)
    return image

In [7]:
image = tf.io.read_file("../images/img1.jpg")
image = tf.image.decode_jpeg(image, channels=3)

image = preprocess_image(image)

output = movenet(image)

In [8]:
output["output_0"].numpy()[0, 0, :, :]

array([[0.26091227, 0.607057  , 0.5519813 ],
       [0.23776537, 0.6330817 , 0.8420506 ],
       [0.24214427, 0.58204335, 0.7260229 ],
       [0.24133453, 0.67932606, 0.78615105],
       [0.25123045, 0.5581516 , 0.5589052 ],
       [0.3725252 , 0.755851  , 0.4841841 ],
       [0.35968685, 0.49778554, 0.67129415],
       [0.5103941 , 0.6991241 , 0.5701693 ],
       [0.47558093, 0.4032417 , 0.2349681 ],
       [0.5117151 , 0.5538759 , 0.49021262],
       [0.505522  , 0.43147072, 0.4613486 ],
       [0.60914415, 0.6154854 , 0.522512  ],
       [0.5881576 , 0.48695773, 0.48249775],
       [0.5518106 , 0.37111902, 0.16629712],
       [0.54044914, 0.3287998 , 0.44198936],
       [0.7891051 , 0.30408245, 0.27566093],
       [0.77560556, 0.32121766, 0.18111445]], dtype=float32)

VIDEO

In [9]:
cap = cv2.VideoCapture(0)

# Keypoint connections
connections = [
    (0, 1), (0, 2),       # nose → eyes
    (1, 3), (2, 4),       # eyes → ears

    (5, 7), (7, 9),       # left shoulder → elbow → wrist
    (6, 8), (8, 10),      # right shoulder → elbow → wrist

    (5, 6),               # shoulders

    (5, 11), (6, 12),     # shoulders → hips
    (11, 12),             # hips

    (11, 13), (13, 15),   # left hip → knee → ankle
    (12, 14), (14, 16)    # right hip → knee → ankle
]

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Get frame dimensions
    height, width, _ = frame.shape

    # Preprocess
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = tf.image.resize(img, (192, 192))
    img = tf.cast(img, tf.int32)
    img = tf.expand_dims(img, axis=0)

    # Run MoveNet
    output = movenet(img)

    # Get keypoints
    keypoints = output["output_0"].numpy()[0, 0, :, :]

    # Draw connections
    for point1, point2 in connections:

        y1, x1, confidence1 = keypoints[point1]
        y2, x2, confidence2 = keypoints[point2]

        # Only draw if both points are confidently detected
        if confidence1 > 0.3 and confidence2 > 0.3:

            x1 = int(x1 * width)
            y1 = int(y1 * height)

            x2 = int(x2 * width)
            y2 = int(y2 * height)

            # White line
            cv2.line(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )

    # Draw circles
    for keypoint in keypoints:

        y, x, confidence = keypoint

        if confidence > 0.3:

            x = int(x * width)
            y = int(y * height)

            # White circle
            cv2.circle(
                frame,
                (x, y),
                5,
                (255, 255, 255),
                -1
            )

    cv2.imshow("MoveNet", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()